# Traffic Pulse AI — optimized retraining pipeline\n\nThis notebook replaces the exploratory notebook/checkpoint pair with one reproducible path. It combines the repository traffic log with a CSV export of the Google Sheet, normalizes timestamp/header differences, removes duplicate `(timestamp, junction)` rows, creates causal lag features, uses a chronological split, and evaluates a retrained Random Forest against a last-value baseline.\n\nIt intentionally does not claim to retrain the proposed GNN: that model's source implementation is not present in the repository yet.

In [ ]:
from pathlib import Path\nimport json\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error\n

In [ ]:
REQUIRED = ['timestamp','junction','lat','lon','current_speed','free_flow_speed','current_travel_time','free_flow_travel_time','confidence','road_closure']\nFEATURES = ['current_speed','free_flow_speed','current_travel_time','free_flow_travel_time','confidence','lat','lon','speed_lag_1','speed_lag_2','speed_lag_3','speed_roll_3','free_flow_ratio','junction_code']\nREPO_CSV = Path('../data/raw/chennai_traffic_log.csv')\nSHEET_CSV = Path('../exports/chennai_traffic_sheet_export.csv')  # export Sheet1 here before running\n

In [ ]:
def load_csv(path, source):\n    frame = pd.read_csv(path)\n    frame.columns = [str(c).strip().lower() for c in frame.columns]\n    if 'time_stamp' in frame and 'timestamp' not in frame:\n        frame = frame.rename(columns={'time_stamp':'timestamp'})\n    missing = sorted(set(REQUIRED) - set(frame.columns))\n    if missing: raise ValueError(f'{path} missing {missing}')\n    frame['source'] = source\n    return frame[REQUIRED + ['source']]\n\nframes = [load_csv(REPO_CSV, 'repository')]\nif SHEET_CSV.exists(): frames.append(load_csv(SHEET_CSV, 'google_sheet'))\ndf = pd.concat(frames, ignore_index=True)\ndf['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True, format='mixed')\ndf['junction'] = df['junction'].astype(str).str.strip()\ndf = df.dropna(subset=['timestamp','junction','current_speed']).drop_duplicates(['timestamp','junction'], keep='last')\ndf = df.sort_values(['timestamp','junction']).reset_index(drop=True)\ndf.source.value_counts()\n

In [ ]:
grouped = df.groupby('junction', group_keys=False)\nfor lag in (1, 2, 3): df[f'speed_lag_{lag}'] = grouped['current_speed'].shift(lag)\ndf['speed_roll_3'] = grouped['current_speed'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())\ndf['free_flow_ratio'] = df['current_speed'] / df['free_flow_speed'].replace(0, np.nan)\ndf['target_speed'] = grouped['current_speed'].shift(-1)\ndf['junction_code'] = pd.Categorical(df['junction']).codes\nmodel_df = df.dropna(subset=FEATURES + ['target_speed']).copy()\n

In [ ]:
cutoff = model_df['timestamp'].quantile(0.80)\ntrain = model_df[model_df.timestamp <= cutoff]\ntest = model_df[model_df.timestamp > cutoff]\nmodel = RandomForestRegressor(n_estimators=120, max_depth=18, min_samples_leaf=2, random_state=42, n_jobs=-1)\nmodel.fit(train[FEATURES], train.target_speed)\nprediction = model.predict(test[FEATURES])\nlast_value = test.speed_lag_1.to_numpy()\n

In [ ]:
def metrics(actual, predicted):\n    return {'MAE': round(mean_absolute_error(actual, predicted), 4), 'RMSE': round(np.sqrt(mean_squared_error(actual, predicted)), 4), 'MAPE': round(np.mean(np.abs((actual-predicted)/np.maximum(np.abs(actual),1.0)))*100, 4)}\nresults = {'last_value': metrics(test.target_speed, last_value), 'random_forest_retrained': metrics(test.target_speed, prediction)}\nprint(json.dumps({'rows':len(df), 'usable_rows':len(model_df), 'cutoff':cutoff.isoformat(), 'sources':df.source.value_counts().to_dict(), 'metrics':results}, indent=2))\nPath('../logs').mkdir(exist_ok=True)\nPath('../logs/retrained_baseline_metrics.json').write_text(json.dumps(results, indent=2))\njoblib.dump({'model':model, 'features':FEATURES, 'cutoff':cutoff.isoformat()}, '../models/retrained_random_forest.joblib')\n